# CPC-based application extraction
This original notebook retrieves application-level rows matching any listed CPC prefix. Because CPC symbols are included, the same application can appear in multiple rows. It does not perform keyword extraction or CPC relevance validation.

For the tutorial-aligned **candidate families → date/language checks → TF-IDF keywords** workflow, open [sector_keywords.ipynb](sector_keywords.ipynb). The descriptive phrase tagging in [cpc_validation.ipynb](cpc_validation.ipynb) is only an optional supplementary diagnostic. Neither workflow automatically validates official CPC definitions or removes CPC codes.

In [22]:

#Import setup and key-search elements
%run setup.ipynb
%run key_words.ipynb

def get_sql_query_by_cpc():
    """SQL query that dynamically get CPC code."""
    
    # Format codes to use them in a SQL LIKE (es. 'G06N%')
    cpc_conditions = " OR ".join([
        f"REPLACE(t224.cpc_class_symbol, ' ', '') LIKE '{code.replace(' ', '')}%'" 
        for code in CPC_CODES
    ])
    
    query_string = f"""
    SELECT 
        DISTINCT
        t201.appln_id,                     --Single patent requests
        t201.appln_auth,                    --Nation
        t224.cpc_class_symbol,             --An appln_id could have more cpc_class_symbol, creates duplicates
        --t201.appln_auth || t201.appln_nr AS appln_num,
        t202.appln_title,
        t203.appln_abstract,
        t201.appln_filing_date,
        --t201.earliest_filing_year,         --Invention date
        t201.docdb_family_id,                --At which technology corresponds the request
        t201.nb_citing_docdb_fam           --How many citation: technological importance

        
    FROM tls201_appln t201
    LEFT JOIN tls224_appln_cpc t224 ON t201.APPLN_ID=t224.APPLN_ID
    LEFT JOIN tls202_appln_title t202 ON t202.appln_id=t201.appln_id
    LEFT JOIN tls203_appln_abstr t203 ON t203.appln_id=t201.appln_id
    WHERE ({cpc_conditions}) 
    AND t201.appln_id < 900000000      --Exclude dummy and fake patents
    ORDER BY t201.appln_filing_date DESC
    
    """
    return text(query_string)

#Get query
sql_query = get_sql_query_by_cpc()

df_results = pd.read_sql(sql_query, db.bind)

display(df_results)

,appln_id,appln_auth,cpc_class_symbol,appln_title,appln_abstract,appln_filing_date,docdb_family_id,nb_citing_docdb_fam
0,601866985,US,G06T 7/70,"METHOD, AERIAL VEHICLE AND SYSTEM FOR DETECTIN...",Embodiments according to a first and second as...,2023-07-20,80122463,0
1,597660526,CN,G06T 7/80,Rotary blurred image deblurring method in blad...,The invention discloses a rotary blurred image...,2023-07-17,87682078,0
2,597487794,CN,G06T 7/70,Aircraft model vibration measurement method ba...,The invention discloses an aircraft model vibr...,2023-07-12,87608415,1
3,601108381,US,H04R 1/028,WILDLIFE DETERRING WINDMILL,Described herein are various systems and techn...,2023-05-01,82900525,1
4,593162322,CN,G06T 7/70,Method and device for determining curve data o...,The invention provides a method and a device f...,2023-04-25,86414970,0
...,...,...,...,...,...,...,...,...
732,54401490,US,G06F 1/1632,Method and apparatus for providing a portable ...,A portable information storage and transfer de...,1992-11-13,25138726,195
733,53793751,US,G02B 6/34,Apparatus for launching or detecting waves of ...,A launcher for waves of selected mode in an op...,1977-11-03,26267496,164
734,53273445,US,G02C 5/146,Adjustable eyeglass temple,A temple rotatably mounted on an eyepiece of a...,1977-01-10,25049706,5
735,22620189,GB,G02B 6/34,APPARATUS FOR LAUNCHING OR DETECTING WAVES OF ...,1527228 Light guides POST OFFICE 10 Dec 1975 [...,1974-12-18,10472011,150
